# Evaluate the 4-bit model vs full precision (Kaggle, T4 x2)

Scores the quantized model (`commit-model-Q4_K_M.gguf`: merged adapter, 4-bit) on the full
CommitBench test split and compares it with the full-precision (fp32) fine-tuned model, to see
whether quantizing cost quality.

The GGUF runs in **llama.cpp** (`llama-server`), the engine Ollama uses, so the scores reflect what
users will actually run. Prompt, truncation and decoding match `evaluate_Kaggle.ipynb`.

**Comparison with fp32:**
- Default: against the fp32 numbers from `results.md` (full test set).
- Better: also attach the output of the `evaluate_Kaggle.ipynb` run (`preds_finetuned.jsonl`). Then
  fp32 is recomputed from its predictions and both models are compared example by example, with a
  significance test (McNemar) on type accuracy.

Output, in the version's Output tab: `preds_q4.jsonl`, `results_q4.json`.

**Setup on Kaggle:**
1. Settings -> Accelerator -> GPU T4 x2, Internet -> On
2. Add Data -> upload `commit-model-Q4_K_M.gguf` as a dataset and attach it, plus the dataset with
   `test.jsonl` (and optionally the `evaluate_Kaggle.ipynb` output, see above)
3. Run with **Save Version -> Save & Run All (Commit)**

In [ ]:
# --- Dependencies ---
# Only the tokenizer (for prompt truncation) and the metrics; the model runs in llama.cpp.
%pip install -q -U transformers evaluate sacrebleu rouge_score

In [ ]:
import glob
import json
import os
import re
import subprocess
import time
import urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import evaluate
import pandas as pd
import torch
from scipy.stats import binomtest
from transformers import AutoTokenizer

# --- Config ---
TOKENIZER_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # same tokenizer as the merged model
LLAMA_TAG = "b11476"     # llama.cpp release used to quantize the model
OUT_DIR = "/kaggle/working"
WORK_DIR = "/tmp/llama"
LIMIT = None             # e.g. 200 for a quick check; None = full test set
SLOTS_PER_GPU = 8        # requests each server processes at once

SEQ_LEN = 768            # must match training
MAX_NEW_TOKENS = 40
PROMPT_TEMPLATE = "Write a Conventional Commits message for this diff.\n\n{diff}\n\nCommit message:"

# fp32 fine-tuned model on the full test set, from results.md. Used when no fp32 predictions are attached.
FP32_RESULTS = {"valid_format": 0.999, "type_acc": 0.673, "BLEU": 19.425, "ROUGE-L": 0.328}
FP32_PER_TYPE = {"fix": 0.862, "chore": 0.377, "feat": 0.364, "test": 0.727, "refactor": 0.345,
                 "docs": 0.746, "build": 0.235, "style": 0.069, "perf": 0.059, "ci": 0.111}
os.makedirs(WORK_DIR, exist_ok=True)


def find_input(pattern, required=True):
    matches = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    if not matches and required:
        raise FileNotFoundError(
            f"Couldn't find {pattern} anywhere under /kaggle/input. Check Add Data attached it. "
            f"Found: {glob.glob('/kaggle/input/**/*', recursive=True)[:50]}"
        )
    return matches[0] if matches else None


def run(cmd, **kwargs):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=True, **kwargs)


GGUF = find_input("*.gguf")
TEST_FILE = find_input("test.jsonl")
FP32_PREDS = find_input("preds_finetuned.jsonl", required=False)
N_GPUS = torch.cuda.device_count()
print("Model:      ", GGUF)
print("Test file:  ", TEST_FILE)
print("fp32 preds: ", FP32_PREDS or "not attached, comparing against results.md numbers")
print("GPUs:       ", N_GPUS)
assert N_GPUS > 0, "Enable a GPU accelerator (Settings -> Accelerator -> GPU T4 x2)"


# --- 1. Get a CUDA build of llama-server ---
def server_env(lib_dirs, gpu=None):
    env = {**os.environ, "LD_LIBRARY_PATH": ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])}
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    return env


def sees_cuda(server_bin, lib_dirs):
    out = subprocess.run([server_bin, "--list-devices"], capture_output=True, text=True, env=server_env(lib_dirs))
    print(out.stdout + out.stderr)
    return "CUDA" in out.stdout


def prebuilt_server():
    # Release binaries: seconds to set up, but need a recent enough NVIDIA driver.
    base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
    dest = f"{WORK_DIR}/prebuilt"
    os.makedirs(dest, exist_ok=True)
    for name in [f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz", f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"]:
        run(["curl", "-sfL", "-o", f"{dest}/{name}", f"{base}/{name}"])
        run(["tar", "-xzf", f"{dest}/{name}", "-C", dest])
    server_bin = glob.glob(f"{dest}/**/llama-server", recursive=True)[0]
    lib_dirs = sorted({os.path.dirname(p) for p in glob.glob(f"{dest}/**/*.so*", recursive=True)})
    return server_bin, lib_dirs


def source_server():
    # Fallback: compile with CUDA for the T4 only (compute capability 7.5). Takes ~15-25 min.
    src = f"{WORK_DIR}/llama.cpp"
    if not os.path.exists(src):
        run(["git", "clone", "--depth", "1", "--branch", LLAMA_TAG, "https://github.com/ggml-org/llama.cpp", src])
    nvcc = "/usr/local/cuda/bin/nvcc"
    run(["cmake", "-S", src, "-B", f"{src}/build", "-DCMAKE_BUILD_TYPE=Release", "-DGGML_CUDA=ON",
         "-DCMAKE_CUDA_ARCHITECTURES=75", "-DLLAMA_BUILD_TESTS=OFF"]
        + ([f"-DCMAKE_CUDA_COMPILER={nvcc}"] if os.path.exists(nvcc) else []))
    run(["cmake", "--build", f"{src}/build", "--target", "llama-server", "-j", str(os.cpu_count())])
    return f"{src}/build/bin/llama-server", [f"{src}/build/bin"]


try:
    SERVER_BIN, LIB_DIRS = prebuilt_server()
    if not sees_cuda(SERVER_BIN, LIB_DIRS):
        raise RuntimeError("prebuilt binaries can't see the GPU (driver too old?)")
except Exception as e:
    print(f"Prebuilt llama.cpp unusable ({e}); building from source.")
    SERVER_BIN, LIB_DIRS = source_server()
    assert sees_cuda(SERVER_BIN, LIB_DIRS), "llama.cpp built without working CUDA"


# --- 2. Start one llama-server per GPU ---
ports = [8090 + gpu for gpu in range(N_GPUS)]
servers = []
for gpu, port in enumerate(ports):
    log = open(f"{WORK_DIR}/server_{gpu}.log", "w")
    servers.append(subprocess.Popen(
        [SERVER_BIN, "-m", GGUF, "-ngl", "99", "-np", str(SLOTS_PER_GPU),
         "-c", str(SLOTS_PER_GPU * 1024), "--port", str(port)],
        env=server_env(LIB_DIRS, gpu), stdout=log, stderr=subprocess.STDOUT,
    ))

for gpu, port in enumerate(ports):
    for _ in range(300):
        try:
            if json.load(urllib.request.urlopen(f"http://localhost:{port}/health")).get("status") == "ok":
                break
        except Exception:
            pass
        if servers[gpu].poll() is not None:
            raise RuntimeError(f"server {gpu} exited:\n" + open(f"{WORK_DIR}/server_{gpu}.log").read()[-3000:])
        time.sleep(1)
    else:
        raise TimeoutError(f"server {gpu} not ready after 5 min")
print("Servers ready on ports", ports)


# --- 3. Generate predictions ---
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
TEMPLATE_TOKENS = len(tokenizer(PROMPT_TEMPLATE.format(diff=""), add_special_tokens=False).input_ids)
DIFF_BUDGET = max(SEQ_LEN - TEMPLATE_TOKENS - MAX_NEW_TOKENS - 8, 0)


def fit_prompt(diff):
    # Truncate the diff, not the prompt's tail, so "Commit message:" always survives.
    diff_ids = tokenizer(diff, add_special_tokens=False).input_ids[:DIFF_BUDGET]
    return PROMPT_TEMPLATE.format(diff=tokenizer.decode(diff_ids))


def generate(i_and_example):
    i, ex = i_and_example
    request = urllib.request.Request(
        f"http://localhost:{ports[i % len(ports)]}/completion",
        headers={"Content-Type": "application/json"},
        data=json.dumps({"prompt": fit_prompt(ex["diff"]), "n_predict": MAX_NEW_TOKENS,
                         "temperature": 0, "top_k": 1}).encode(),  # greedy, as in the fp32 eval
    )
    text = json.load(urllib.request.urlopen(request, timeout=300))["content"]
    return (text.strip().splitlines() or [""])[0].strip()


with open(TEST_FILE) as f:
    examples = [json.loads(line) for line in f][:LIMIT]

started = time.time()
preds = []
with ThreadPoolExecutor(max_workers=SLOTS_PER_GPU * len(ports)) as pool:
    for pred in pool.map(generate, enumerate(examples)):
        preds.append(pred)
        if len(preds) % 500 == 0 or len(preds) == len(examples):
            print(f"[{(time.time() - started) / 60:5.1f} min] {len(preds)}/{len(examples)}", flush=True)

for server in servers:
    server.terminate()

refs = [ex["message"] for ex in examples]
with open(f"{OUT_DIR}/preds_q4.jsonl", "w") as f:
    for pred, ref in zip(preds, refs):
        f.write(json.dumps({"pred": pred, "ref": ref}) + "\n")


# --- 4. Metrics: 4-bit vs fp32 ---
TYPE_RE = re.compile(r"^(\w+)(\(.+?\))?!?:")
VALID_RE = re.compile(r"^(feat|fix|refactor|chore|docs|test|perf|style|build|ci)(\([\w./-]+\))?!?:\s+\S")


def extract_type(message):
    match = TYPE_RE.match(message.strip())
    return match.group(1) if match else None


bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")
ref_types = [extract_type(r) for r in refs]


def score(p):
    return {
        "valid_format": sum(bool(VALID_RE.match(x)) for x in p) / len(p),
        "type_acc": sum(extract_type(x) == t for x, t in zip(p, ref_types)) / len(p),
        "BLEU": bleu.compute(predictions=p, references=[[r] for r in refs])["score"],
        "ROUGE-L": rouge.compute(predictions=p, references=refs)["rougeL"],
    }


def per_type(p):
    return {t: sum(extract_type(x) == t for x, rt in zip(p, ref_types) if rt == t) / n
            for t, n in Counter(ref_types).most_common()}


fp32_preds = None
if FP32_PREDS:
    with open(FP32_PREDS) as f:
        fp32_records = [json.loads(line) for line in f][: len(examples)]
    if [r["ref"] for r in fp32_records] == refs:
        fp32_preds = [r["pred"] for r in fp32_records]
    else:
        print("Attached fp32 predictions don't line up with this test set; using results.md numbers instead.")

if fp32_preds is None and LIMIT is not None:
    print("Note: results.md fp32 numbers are for the full test set; with LIMIT set, compare loosely.")

q4 = score(preds)
fp32 = score(fp32_preds) if fp32_preds else FP32_RESULTS
results = pd.DataFrame({"fp32": fp32, "Q4_K_M": q4}).T
results.loc["change"] = results.loc["Q4_K_M"] - results.loc["fp32"]
print(f"\nExamples: {len(refs)}\n")
print(results.to_string(float_format=lambda v: f"{v:.3f}"))

q4_types = per_type(preds)
fp32_types = per_type(fp32_preds) if fp32_preds else FP32_PER_TYPE
print("\nType accuracy by type:")
print(f"  {'type':<10}{'n':>6}{'fp32':>9}{'Q4_K_M':>9}{'change':>9}")
for t, n in Counter(ref_types).most_common():
    a, b = fp32_types.get(t, float("nan")), q4_types[t]
    print(f"  {t:<10}{n:>6}{a:>9.1%}{b:>9.1%}{b - a:>+9.1%}")

# Is the type-accuracy change real or noise?
if fp32_preds:
    fp32_ok = [extract_type(x) == t for x, t in zip(fp32_preds, ref_types)]
    q4_ok = [extract_type(x) == t for x, t in zip(preds, ref_types)]
    only_fp32 = sum(a and not b for a, b in zip(fp32_ok, q4_ok))
    only_q4 = sum(b and not a for a, b in zip(fp32_ok, q4_ok))
    p_value = binomtest(only_fp32, only_fp32 + only_q4, 0.5).pvalue if only_fp32 + only_q4 else 1.0
    same_message = sum(a == b for a, b in zip(fp32_preds, preds)) / len(preds)
    print(f"\nIdentical messages, fp32 vs Q4: {same_message:.1%}")
    print(f"Type right only in fp32: {only_fp32}, only in Q4: {only_q4} (McNemar p = {p_value:.3f})")
    print("Verdict:", "significant change" if p_value < 0.05 else "no significant change", "in type accuracy")
else:
    p1, p2, n = fp32["type_acc"], q4["type_acc"], len(refs)
    margin = 1.96 * ((p1 * (1 - p1) + p2 * (1 - p2)) / n) ** 0.5
    print(f"\nType accuracy change: {p2 - p1:+.1%} (95% margin ±{margin:.1%}; "
          "attach fp32 predictions for a sharper, example-by-example test)")

print("\nSample predictions:")
for i in range(10):
    print(f"  ref:    {refs[i]}")
    if fp32_preds:
        print(f"  fp32:   {fp32_preds[i]}")
    print(f"  Q4_K_M: {preds[i]}\n")

with open(f"{OUT_DIR}/results_q4.json", "w") as f:
    json.dump({"examples": len(refs), "fp32": fp32, "Q4_K_M": q4,
               "fp32_per_type": fp32_types, "Q4_K_M_per_type": q4_types,
               "fp32_source": "predictions" if fp32_preds else "results.md"}, f, indent=2)